### 1. Write SQL statements to create:

In [0]:
%sql
-- Creating a catalog
create catalog if not exists telecom_catalog_assign;

-- Creating a schema
create schema if not exists telecom_catalog_assign.landing_zone;

-- Creating a volume
create volume if not exists telecom_catalog_assign.landing_zone.landing_vol; 


In [0]:
### Create Folders 
folders = ["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/", "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/", "/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/"]

for i in folders:
    dbutils.fs.mkdirs(i)

print("Folders created successfully.")

### a. Volumes vs DBFS/FileStore
DBFS/FileStore is Databricks' legacy distributed file system abstraction layered on top of cloud object storage. Key traits:

Workspace-scoped, not governed — DBFS root storage lives in a single storage account tied to the workspace, outside Unity Catalog's governance model.
No fine-grained access control — Historically, anyone with cluster access to DBFS root could read/write files; permissions are managed at the workspace/cluster level, not per-object.
FileStore specifically is a special DBFS folder meant for small files (images, libraries) served via a web endpoint — never intended for production data.
No lineage or audit trail — Because it sits outside Unity Catalog, reads/writes to DBFS aren't tracked in Unity Catalog's audit logs or lineage graph.
Path-based, ungoverned access — Data is referenced by raw paths (dbfs:/...), so there's no catalog/schema structure, no access review, no centralized permission model.

Unity Catalog Volumes are a governed abstraction for non-tabular data (files: images, PDFs, JSON, model artifacts, CSVs, etc.):

Fully governed by Unity Catalog — Volumes live inside a catalog/schema hierarchy (catalog.schema.volume), just like tables, and inherit UC's centralized permission model (GRANT, REVOKE at volume level).
Fine-grained access control — You can grant READ VOLUME / WRITE VOLUME to specific users, groups, or service principals, independent of cluster or workspace access.
Audited and lineage-tracked — All access is logged in Unity Catalog audit logs; file-level lineage is captured, so you can trace what job/notebook touched what file.
Cross-workspace governance — Volumes are metastore-level objects, so the same governance applies consistently whether accessed from any workspace attached to that metastore.
Managed or external — Like tables, volumes can be UC-managed (storage handled by UC) or external (pointing to your own cloud storage path, but still governed by UC permissions).

### b. Why production/regulated teams prefer Volumes

Centralized, auditable access control — Regulated industries (finance, healthcare, insurance) need to prove who accessed what data and when. Volumes give per-principal grants and audit logs; DBFS gives none of this — access is essentially "anyone on a cluster with DBFS mount access."
Data lineage for compliance — Regulations like GDPR, HIPAA, SOX often require demonstrable lineage (where did this file come from, what pipeline touched it, who downloaded it). Unity Catalog lineage graphs capture this automatically for Volumes; DBFS paths are invisible to lineage tracking.
Principle of least privilege — With Volumes you can scope access down to a specific schema or even individual files/directories for a specific team or service principal. DBFS's workspace-level access model makes it very hard to prevent over-broad access (a classic audit finding).
Separation of governance from compute — Access to a Volume doesn't depend on which cluster you're using or its permissions mode; it's enforced at the catalog layer regardless of compute. This avoids the DBFS problem where cluster misconfiguration (e.g., a high-concurrency cluster with broad DBFS mounts) becomes a data leak vector.
Consistent governance across structured + unstructured data — Regulated data pipelines often mix tables and files (e.g., a table of transactions + PDF statements). Volumes let you govern the unstructured side with the same catalog/schema/grant model as tables, so security reviews don't need two separate mental models.
Deprecation trajectory — Databricks has been actively steering customers off DBFS root for data storage (it's now largely reserved for temp/scratch use), since it doesn't fit UC's governance-first design. Production/regulated workloads adopting Volumes is also just aligning with where the platform is heading, avoiding technical debt.

In [0]:
from pyspark.sql.functions import col
# 3. Spark Directory Read Use Cases

telecom_df=spark.read.option("header","true").option("recursiveFileLookup", "true").option("pathGlobFilter","*.csv").csv(["dbfs:/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer","dbfs:/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage","dbfs:/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower"])

print("With recursiveFileLookup   :", telecom_df.count())

telecom_df.select("*",col("_metadata.file_path").alias("Source File")).distinct().show(truncate=False)

In [0]:
# 2. Filesystem operations
display(dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer"))
display(dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage"))
display(dbutils.fs.ls("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower"))





In [0]:
# 3. Spark Directory Read Use Cases
